## FFC — Step 3 (Local): Apollo + Keimed DB-level

This notebook is a local execution adaptation of `FFC/Step 3 - Keimed_Dev-Apollo_DB Level-2026.py`.

**Business logic is unchanged** (same joins, filters, renames). Only S3 listing/paths are replaced by local folders.

### Expected local folder layout

Place the same input files you normally read from S3 into these local folders:

- `input/keimed_gt/` (raw Keimed xlsx/xlsb files)
- `input/apollo_keimed/` (raw Apollo xlsx/xlsb files)
- `input/static_keimed/Keimed DBs Data Sharing.xlsx`
- `input/static_keimed/2026_Kiemed_Product_list.xlsx`
- `input/2026_SKU_Brand_Division_mapping.xlsx`
- `input/2026_PinCodes_with_States_and_Districts.csv`

### Output
- `output/DB_Level_Apollo_Keimed_2026_local.csv`
- `output/202603_Apollo_Keimed_DB_Level_QC_local.xlsx`


In [25]:
import os
import glob
import pandas as pd
import numpy as np

os.makedirs("output", exist_ok=True)

KEIMED_DIR = "input/keimed_gt"
APOLLO_DIR = "input/apollo_gt"

KEIMED_DB_MAP_FILE = "input/Keimed DBs Data Sharing.xlsx"
KEIMED_PRODUCT_FILE = "input/2026_Kiemed_Product_list.xlsx"

SKU_BRAND_DIV_FILE = "input/2026_SKU_Brand_Division_mapping.xlsx"
PINCODE_FILE = "input/2026_PinCodes_with_States_and_Districts.csv"

OUTPUT_DB_LEVEL = "output/DB_Level_Apollo_Keimed_2026_local.csv"
OUTPUT_QC = "output/202603_Apollo_Keimed_DB_Level_QC_local.xlsx"


In [26]:
# Helpers

def list_files(folder: str):
    patterns = ["*.xlsx", "*.xlsb", "*.csv"]
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(folder, p)))
    files = sorted(files)
    return files

def read_excel_any(path: str, **kwargs):
    if path.lower().endswith(".xlsb"):
        return pd.read_excel(path, engine="pyxlsb", **kwargs)
    return pd.read_excel(path, **kwargs)


In [27]:

# Load reference masters

sku_map = pd.read_excel(SKU_BRAND_DIV_FILE)
pin_city_state_mapping_data = pd.read_csv(PINCODE_FILE, encoding="unicode_escape")

keimed_cust = pd.read_excel(KEIMED_DB_MAP_FILE, sheet_name="DBs")
keimed_cust2 = keimed_cust[~pd.isna(keimed_cust["Keimed Code"])].copy()
keimed_cust2 = keimed_cust2[~pd.isna(keimed_cust2["Cust. Code"])].copy()
keimed_cust2 = keimed_cust2[["Cust. Code", "Keimed Code", "Cust. Name"]]

keimed_prod = pd.read_excel(KEIMED_PRODUCT_FILE, sheet_name="Mapped Abbott Sku")
keimed_prod2 = keimed_prod[keimed_prod["Exclusion"] != "Exclude"].copy()
keimed_prod2 = keimed_prod2[["Sku", "Apollo_Mapped"]]

keimed_prod3 = pd.merge(
    keimed_prod2,
    sku_map[["SKU Code", "Division Name", "Affiliate"]],
    how="left",
    left_on="Apollo_Mapped",
    right_on="SKU Code",
)
keimed_prod3 = keimed_prod3[["Sku", "Apollo_Mapped", "Division Name", "Affiliate"]]
keimed_prod3["Division Name"] = np.where(pd.isna(keimed_prod3["Division Name"]), "Not Found", keimed_prod3["Division Name"])
keimed_prod3["Affiliate"] = np.where(pd.isna(keimed_prod3["Affiliate"]), "Not Found", keimed_prod3["Affiliate"])

{
    "sku_map": sku_map.shape,
    "pin_map": pin_city_state_mapping_data.shape,
    "keimed_cust2": keimed_cust2.shape,
    "keimed_prod3": keimed_prod3.shape,
}


{'sku_map': (2492, 6),
 'pin_map': (20873, 3),
 'keimed_cust2': (88, 3),
 'keimed_prod3': (1427, 4)}

In [28]:
# Keimed ingest


keimed_files = list_files(KEIMED_DIR)
keimed_files[:5], len(keimed_files)


(["input/keimed_gt\\Abbott_Secondary sales Mar'26.xlsb"], 1)

In [29]:
cols_to_keep = [
    "month",
    "DB code",
    "DB location",
    "DB city",
    "sku_id",
    "sku_name",
    "customer_pincode",
    "Brand_SKU units sold",
    "Brand_SKU GMV",
]

dfs = []
for file in keimed_files:
    try:
        df_inter = read_excel_any(file)
        # df_inter.columns = df_inter.columns.str.strip().str.lower()
        # same business logic as script: Excel serial date conversion
        df_inter["month"] = pd.to_datetime(df_inter["month"], unit="D", origin="1899-12-30")
        df_inter = df_inter[cols_to_keep]
        dfs.append(df_inter)
        print(os.path.basename(file), "Done")
    except Exception as e:
        print("FAILED:", os.path.basename(file), "|", e)

data_keimed = pd.concat(dfs, ignore_index=True)
data_keimed.shape


Abbott_Secondary sales Mar'26.xlsb Done


(263760, 9)

In [30]:
# keimed customer code mapping
data_keimed_2 = pd.merge(
    data_keimed,
    keimed_cust2,
    how="left",
    left_on="DB code",
    right_on="Keimed Code",
)

data_keimed_2 = data_keimed_2[~pd.isna(data_keimed_2["Cust. Code"])].copy()

# keimed product mapping
data_keimed_3 = pd.merge(
    data_keimed_2,
    keimed_prod3,
    how="left",
    left_on="sku_id",
    right_on="Sku",
)

data_keimed_3 = data_keimed_3[~pd.isna(data_keimed_3["Apollo_Mapped"])].copy()
data_keimed_3.shape


(212213, 16)

In [31]:
# Build Keimed output schema
data_kiemed_4 = data_keimed_3[
    [
        "month",
        "Cust. Code",
        "customer_pincode",
        "Cust. Name",
        "Apollo_Mapped",
        "Brand_SKU GMV",
        "Division Name",
        "Affiliate",
    ]
].copy()

data_kiemed_4["Zone"] = 0
data_kiemed_4["CFA Name"] = 0
data_kiemed_4["name of customer"] = "KiemedGT"
data_kiemed_4["ECB_Primary_Sales"] = 0
data_kiemed_4["units_sold"] = 0
data_kiemed_4["PTS"] = 0

# merge pin-state
data_kiemed_5 = pd.merge(
    left=data_kiemed_4,
    right=pin_city_state_mapping_data,
    how="left",
    left_on=["customer_pincode"],
    right_on=["Pin.Code"],
)

# IMPORTANT: replicate original Step-3 positional rename
data_kiemed_5.rename(columns={
    data_kiemed_5.columns[0]: 'month',
    data_kiemed_5.columns[1]: 'Customer Code',
    data_kiemed_5.columns[2]: 'pincode',
    data_kiemed_5.columns[3]: 'Customer Name (Sold to Party)',
    data_kiemed_5.columns[4]: 'SKU No',
    data_kiemed_5.columns[5]: 'channel_sales'
}, inplace=True)

data_kiemed_6 = pd.merge(
    data_kiemed_5,
    sku_map[["SKU Code", "Merged SKU Code", "Brand Name", "SKU Name"]],
    how="left",
    left_on="SKU No",
    right_on="SKU Code",
)

data_kiemed_7 = data_kiemed_6[
    [
        "month",
        "Customer Code",
        "pincode",
        "Zone",
        "Customer Name (Sold to Party)",
        "CFA Name",
        "name of customer",
        "SKU No",
        "channel_sales",
        "ECB_Primary_Sales",
        "units_sold",
        "PTS",
        "Merged SKU Code",
        "Division Name",
        "Brand Name",
        "SKU Name",
        "Affiliate",
        "State",
        "District",
    ]
].copy()

data_kiemed_7.shape


(212213, 19)

In [32]:
#  Apollo ingest


apollo_files = list_files(APOLLO_DIR)
apollo_files[:5], len(apollo_files)


(["input/apollo_gt\\Abbott - Mar'2026.xlsb"], 1)

In [33]:
cols_to_keep_apollo = [
    "GOLD CODE",
    "PRODUCT DESCRIPTION - APOLLO",
    "PIN CODE",
    "QTY",
    "VALUE",
    "SUPPLIER",
]

month_list = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

dfs_ap = []
for file in apollo_files:
    try:
        df_inter = read_excel_any(file, sheet_name=0)
        df_inter = df_inter[cols_to_keep_apollo]
        # same logic: infer month from filename patterns like "mar'2026"
        m = [x.upper() for x in month_list if (x + "'2026") in os.path.basename(file).lower()]
        if len(m) == 0:
            raise ValueError("Cannot infer month from filename")
        df_inter["month"] = pd.to_datetime("01" + m[0] + "2026", format="%d%b%Y")
        dfs_ap.append(df_inter)
        print(os.path.basename(file), "Done")
    except Exception as e:
        print("FAILED:", os.path.basename(file), "|", e)

data_apollo = pd.concat(dfs_ap, ignore_index=True)
data_apollo.shape


Abbott - Mar'2026.xlsb Done


(869412, 7)

In [34]:
# Apollo product mapping (same as script)
data_apollo_3 = pd.merge(
    data_apollo,
    keimed_prod3,
    how="left",
    left_on="GOLD CODE",
    right_on="Sku",
)
data_apollo_3 = data_apollo_3[~pd.isna(data_apollo_3["Apollo_Mapped"])].copy()

data_apollo_3["Cust. Code"] = 0
data_apollo_3["Zone"] = 0
data_apollo_3["CFA Name"] = 0
data_apollo_3["name of customer"] = "Apollo"
data_apollo_3["ECB_Primary_Sales"] = 0
data_apollo_3["units_sold"] = 0
data_apollo_3["PTS"] = 0

# merge pin-state
data_apollo_5 = pd.merge(
    left=data_apollo_3,
    right=pin_city_state_mapping_data,
    how="left",
    left_on=["PIN CODE"],
    right_on=["Pin.Code"],
)

# drop columns per script
data_apollo_5.drop(["PRODUCT DESCRIPTION - APOLLO", "GOLD CODE", "QTY", "Sku"], axis=1, inplace=True)

# IMPORTANT: replicate original Step-3 positional rename
data_apollo_5.rename(columns={
    data_apollo_5.columns[2]:'Customer Name (Sold to Party)',
    data_apollo_5.columns[3]:'month',
    data_apollo_5.columns[0]:'pincode',
    data_apollo_5.columns[4]:'SKU No',
    data_apollo_5.columns[1]:'channel_sales'
}, inplace=True)

# match original
data_apollo_5.drop(["Pin.Code"], axis=1, inplace=True)

data_apollo_5["name of customer"] = data_apollo_5["name of customer"].str.lower()

data_apollo_6 = pd.merge(
    data_apollo_5,
    sku_map[["SKU Code", "Merged SKU Code", "Brand Name", "SKU Name"]],
    how="left",
    left_on="SKU No",
    right_on="SKU Code",
)

# match original positional rename
data_apollo_6.rename(columns={data_apollo_6.columns[7]: "Customer Code"}, inplace=True)

data_apollo_7 = data_apollo_6[
    [
        "month",
        "Customer Code",
        "pincode",
        "Zone",
        "Customer Name (Sold to Party)",
        "CFA Name",
        "name of customer",
        "SKU No",
        "channel_sales",
        "ECB_Primary_Sales",
        "units_sold",
        "PTS",
        "Merged SKU Code",
        "Division Name",
        "Brand Name",
        "SKU Name",
        "Affiliate",
        "State",
        "District",
    ]
].copy()

data_apollo_7.shape


(869323, 19)

In [35]:
# -----------------------------------------------------
# Combine + write outputs
# -----------------------------------------------------

Apollo_keimed = pd.concat([data_kiemed_7, data_apollo_7], ignore_index=True)

Apollo_keimed.to_csv(OUTPUT_DB_LEVEL, index=False)

channelqc = (
    Apollo_keimed.groupby(["name of customer", "Customer Name (Sold to Party)", "month"])
    .agg(Channel_Sales=("channel_sales", "sum"), ECB_Primary_Sales=("ECB_Primary_Sales", "sum"))
    .reset_index()
)

channeldivqc = (
    Apollo_keimed.groupby(["name of customer", "Customer Name (Sold to Party)", "Division Name", "month", "State"])
    .agg(Channel_Sales=("channel_sales", "sum"), ECB_Primary_Sales=("ECB_Primary_Sales", "sum"))
    .reset_index()
)

with pd.ExcelWriter(OUTPUT_QC) as writer:
    channelqc.to_excel(writer, sheet_name="Account Sales", index=False)
    channeldivqc.to_excel(writer, sheet_name="Account_Division Sales", index=False)

{
    "db_level_csv": OUTPUT_DB_LEVEL,
    "qc_xlsx": OUTPUT_QC,
    "rows": Apollo_keimed.shape[0],
}


{'db_level_csv': 'output/DB_Level_Apollo_Keimed_2026_local.csv',
 'qc_xlsx': 'output/202603_Apollo_Keimed_DB_Level_QC_local.xlsx',
 'rows': 1081536}